# Module `graph_generator.py`

Ce notebook explique pas a pas comment une instance CESIPATH est construite et pourquoi chaque etape existe.

Le generateur produit quatre objets consommes par le reste du projet :

1. le **graphe de base** : toutes les vraies routes physiques, sans contrainte ;
2. le **graphe residuel** : le meme graphe apres application des contraintes statiques (interdictions, surcouts) ;
3. le **graphe complete** : la matrice de couts obtenue par Dijkstra sur le residuel, utilisable directement par un solveur metrique ;
4. les **snapshots dynamiques** : etats du reseau a chaque tour de simulation, recalcules par `DynamicNetworkSimulator`.

La cle de l'implementation : le generateur garantit la **connexite** et les **bornes de densite** par construction + rejet, jamais par post-traitement destructif.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_network import DynamicNetworkSimulator
from cesipath.graph_generator import GraphGenerator
from cesipath.metric_closure import check_triangle_inequality
from cesipath.models import EdgeStatus, GraphGenerationConfig

## 1. Parametres de generation

`GraphGenerationConfig` regroupe tous les leviers. Les valeurs par defaut suffisent la plupart du temps.

Parametres statiques cles :

- `node_count` : nombre de sommets (depot inclus).
- `seed` : graine de reproductibilite. Deux `generate()` avec la meme graine donnent la meme instance.
- `auto_density_profile=True` : choisit les bornes de densite selon `node_count` (graphes denses quand petits, creux quand grands).
- `forbidden_rate` : proba qu'une arete hors arbre couvrant soit statiquement interdite.
- `surcharge_rate` : proba qu'une arete recoive un surcout statique dans `[surcharge_min, surcharge_max]`.
- `vehicle_capacity`, `demand_min`, `demand_max` : donnent la contrainte de capacite du VRP.

Parametres dynamiques cles (utilises seulement par `DynamicNetworkSimulator`, pas par `generate()`) :

- `dynamic_sigma`, `dynamic_mean_reversion_strength`, `dynamic_max_multiplier` : loi d'evolution des couts ;
- `dynamic_forbid_probability`, `dynamic_restore_probability` : coupures/retablissements temporaires ;
- `dynamic_max_disabled_ratio` : part maximale d'aretes OFF tolerees a un instant donne.

In [ ]:
config = GraphGenerationConfig(
    node_count=7,
    seed=42,
    auto_density_profile=True,
    forbidden_rate=0.1,
    surcharge_rate=0.25,
    vehicle_capacity=40,
)

generator = GraphGenerator(config)
instance = generator.generate()

pprint(instance.summary())

## 2. Etape 1 - Coordonnees uniformes

`_generate_coordinates` tire chaque sommet uniformement dans le rectangle `[0, width] x [0, height]`.
Les couts d'aretes utilisent la distance euclidienne (`math.dist`), arrondie a 2 decimales.

Pourquoi euclidien ? L'inegalite triangulaire est triviale entre points du plan, ce qui facilite la verification de la fermeture metrique plus tard.

In [ ]:
print("Coordonnees :")
pprint(instance.coordinates)

## 3. Etape 2 - Arbre couvrant pour garantir la connexite

Dans `_build_connected_adjacency`, on commence par attacher chaque nouveau sommet `i >= 1` a un parent aleatoire `< i`. Ce procede produit toujours un **arbre couvrant** : tous les sommets sont connexes avant meme d'ajouter d'autres aretes.

C'est la premiere garantie structurelle : aucun tirage aleatoire ne peut produire un graphe deconnecte.

In [ ]:
from cesipath.validators import is_connected

active = set(instance.base_edges.keys())
print("Nombre d'aretes base :", len(active))
print("Graphe de base connexe ?", is_connected(instance.node_count, active))

## 4. Etape 3 - Densification jusqu'a la cible

Une fois l'arbre place, on ajoute des aretes aleatoires jusqu'a atteindre `target_edges`, deduit de `resolved_edge_density`.

Cette densification sert uniquement au graphe de **base**. Elle ne modifie pas la connexite (on ne retire jamais d'arete) et donne au solveur plus d'alternatives de chemins.

In [ ]:
max_edges = instance.node_count * (instance.node_count - 1) // 2
print("Max d'aretes possibles :", max_edges)
print("Densite cible :", config.resolved_edge_density)
print("Densite base obtenue :", round(instance.base_density, 4))

## 5. Etape 4 - Arbre protege contre les contraintes statiques

C'est **le point le plus important** du generateur. Dans `_build_residual_edges`, on applique les tirages `FORBIDDEN` et `SURCHARGED`, **sauf sur les aretes de l'arbre couvrant**. Ces aretes restent toujours `FREE`.

Consequence : meme avec un `forbidden_rate` eleve, le graphe residuel reste connexe. C'est ce qui permet a Dijkstra de toujours trouver un chemin entre deux sommets apres filtrage des aretes interdites.

Sans cette protection, le generateur passerait son temps a etre rejete par `InstanceValidator` et `generation_max_attempts` serait depasse des que les contraintes sont un peu agressives.

In [ ]:
tree_edges = GraphGenerator._spanning_tree_edges(
    {key: edge.base_cost for key, edge in instance.base_edges.items()}
)
tree_with_status = [
    (key, instance.residual_edges[key].status.value) for key in tree_edges
]
print("Aretes de l'arbre couvrant et leur statut :")
pprint(tree_with_status)

non_free_in_tree = [k for k, status in tree_with_status if status != "libre"]
print("\nAretes de l'arbre NON libres (doit etre vide) :", non_free_in_tree)

## 6. Etape 5 - Trois matrices de couts

Le generateur construit trois matrices distinctes :

- **`base_costs`** : graphe physique initial, sans contrainte. Cellule a 0 si pas d'arete directe.
- **`residual_costs`** : apres application des surcouts statiques ; les aretes FORBIDDEN sont retirees (valeur 0).
- **`completed_costs`** : matrice complete obtenue par Dijkstra sur le residuel. Cellule `(i, j)` = cout du plus court chemin de `i` a `j`.

Les solveurs consomment uniquement `completed_costs` (via `SolverInput`). Les autres matrices sont utiles pour le debug et la visualisation.

In [ ]:
def show(title, matrix):
    print(title)
    for row in matrix:
        print(row)
    print()

show("base_costs :", instance.base_costs)
show("residual_costs :", instance.residual_costs)
show("completed_costs :", instance.completed_costs)

## 7. Etape 6 - Fermeture metrique (Dijkstra)

`complete_graph_with_shortest_paths` (dans `metric_closure.py`) applique Dijkstra depuis chaque source et produit deux choses :

- la matrice `completed_costs` ;
- un dictionnaire `completed_paths[(i, j)]` qui contient le vrai chemin passant par les vraies aretes.

Ces chemins reels sont importants : un solveur peut proposer l'arete virtuelle `(0, 3)`, mais le vrai deplacement correspond peut-etre a `0 -> 4 -> 3`. C'est utile pour la visualisation et pour l'execution dynamique (`execute_dynamic`).

In [ ]:
for target in range(1, instance.node_count):
    path = instance.completed_paths.get((0, target))
    cost = instance.completed_costs[0][target]
    print(f"0 -> {target} : cout={cost} via {path}")

## 8. Inegalite triangulaire

Puisque `completed_costs` est construit a partir de plus courts chemins, il doit respecter :

```text
d(i, j) <= d(i, k) + d(k, j)  pour tout triplet (i, j, k)
```

Si cette propriete ne tient pas, les algorithmes metriques (ex. 2-opt, Christofides) perdent leurs garanties. C'est donc un garde-fou important.

In [ ]:
ok, violation = check_triangle_inequality(instance.completed_costs)
print("Inegalite triangulaire respectee ?", ok)
print("Violation eventuelle :", violation)

## 9. Demandes uniformes

`_generate_demands` tire **une seule** valeur entiere dans `[demand_min, demand_max]` et l'affecte a tous les clients. Le depot garde `0`.

Pourquoi uniforme ? Le livrable 1 fixe cette hypothese pour simplifier l'analyse : toutes les livraisons pesent pareil. Le nombre minimal de tournees necessaires se lit alors directement : `ceil(n_clients * demande / capacite)`.

In [ ]:
print("Demande uniforme :", instance.uniform_demand)
print("Capacite vehicule :", config.vehicle_capacity)
print("Nombre minimal de tournees :", instance.minimum_route_count)

## 10. Boucle de rejet (rejection sampling)

`generate()` construit une instance candidate, puis la passe a `InstanceValidator.is_valid`. Quatre criteres doivent etre respectes :

1. densite de base dans `[resolved_min_base_density, resolved_max_base_density]` ;
2. densite residuelle dans `[resolved_min_residual_density, resolved_max_residual_density]` ;
3. degre moyen residuel `>= resolved_min_average_residual_degree` ;
4. graphe residuel connexe (apres retrait des FORBIDDEN).

Si un critere echoue, on recommence avec une nouvelle graine interne, jusqu'a `generation_max_attempts` tentatives. Si rien ne passe, une `ValueError` est levee : c'est le signe que les bornes demandees sont infaisables pour cette taille.

La cellule suivante force un echec volontaire pour montrer le comportement.

In [ ]:
from cesipath.models import GraphGenerationConfig

impossible = GraphGenerationConfig(
    node_count=5,
    auto_density_profile=False,
    min_residual_density=0.95,
    max_residual_density=1.0,
    generation_max_attempts=5,
    seed=0,
)

try:
    GraphGenerator(impossible).generate()
except ValueError as err:
    print("Rejet attendu :", err)

## 11. Passage a la dynamique

Une fois l'instance statique validee, la simulation dynamique prend le relais via `DynamicNetworkSimulator`.

Un `DynamicGraphSnapshot` represente l'etat du reseau a un tour `t`. Il stocke :

- `edge_costs` : cout courant de chaque arete ;
- `edge_availability` : booleen par arete (ON/OFF temporairement) ;
- `residual_costs`, `completed_costs`, `completed_paths` : recalcules apres chaque tour.

`advance()` applique tirage + validation + recalcul. Voir `dynamic_network.ipynb` pour le detail.

In [ ]:
simulator = DynamicNetworkSimulator(instance, seed=42)
snapshot = simulator.initialize_snapshot()
print("Tour initial :", snapshot.step)
print("Aretes actives :", snapshot.active_edge_count)

next_snapshot = simulator.advance(snapshot)
print("Tour suivant :", next_snapshot.step)
print("Aretes actives :", next_snapshot.active_edge_count)
print("\nNouvelle matrice complete ligne 0 :", next_snapshot.completed_costs[0])